## Set the experiment

In [ ]:
import os
class settings:
  def __init__(self):
    self.dataset = "INA"
    self.iter=10
    self.results_dir="experiment_main"
    #self.methods = [['Med2Vec'], ['Dipole'], ['BiPadLSTM'], ['GRU']]  # for end-to-end experiment
    self.methods = [['STATIC'],
                    ['STATIC', 'BINARY'],
                    ['STATIC', 'FREQUENCY'],
                    ['STATIC', 'EVENT-CNT'], 
                    ['STATIC', 'TF-IDF'],
                    ['STATIC', 'BOW'],
                    ['STATIC', 'BiPadLSTM'],
                    ['STATIC', 'Med2Vec'],
                    ['STATIC', 'Dipole'],
                    ['STATIC', 'CEHR-BERT'],
                    ['STATIC', 'DOME'],
                    ['STATIC', 'GRU'],
                    ]

args = settings()

## Download files from Github

In [ ]:
try:
    import google.colab
    %mkdir args.results_dir
    !wget -np https://raw.githubusercontent.com/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/main/{args.results_dir}/* -O {args.results_dir}
    print("Working on Colab!")
except:
    print("Not Working on Colab ... or cannot dwnload Githb files!")
    pass

## Load results

In [2]:
import pickle
# Caricamento dei risultati
results = {"+".join(m): {} for m in args.methods}

for i in range(args.iter):
    for m in args.methods:
        meth = "+".join(m)
        path = (
            f"{args.results_dir}/results_{args.dataset}_"
            f"{'_'.join(m)}_ITER{i}.pkl"
        )
        with open(path, "rb") as f:
            results[meth][i] = pickle.load(f)

## Compute metrics means, standard deviations and CI intervals (with boost)

In [3]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    roc_auc_score,
)


METRIC_NAMES = [
    "AUC", "AUPRC", "F1", "PPV", "Sensitivity",
    "Specificity", "NPV", "MCC", "Accuracy", "Brier",
]


def metric_values(y, p, threshold=0.30):
    """Metrics in a repetition, one value per subject."""
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    return np.array([
        roc_auc_score(y, p),
        average_precision_score(y, p),
        f1_score(y, pred, zero_division=0),
        precision_score(y, pred, zero_division=0),
        tp / (tp + fn),                         # Sensitivity
        tn / (tn + fp),                         # Specificity
        tn / (tn + fn) if (tn + fn) else 0.0,  # NPV
        matthews_corrcoef(y, pred),
        accuracy_score(y, pred),
        brier_score_loss(y, p),
    ])


def align_oof_runs(runs):
    """
    returns y (subjects,) e P (subjects, repetitions).
    """
    ids = pd.Index(runs[0]["y_idx"])
    if not ids.is_unique:
        raise ValueError("ID duplicati nella prima ripetizione")

    y = pd.Series(
        runs[0]["y_true_oof"], index=ids
    ).to_numpy()

    P = np.empty((len(ids), len(runs)), dtype=float)

    for r, run in enumerate(runs):
        run_ids = pd.Index(run["y_idx"])

        if (
            not run_ids.is_unique
            or len(run_ids) != len(ids)
            or not run_ids.difference(ids).empty
        ):
            raise ValueError(f"ID do not match, repetition {r}")

        y_r = pd.Series(
            run["y_true_oof"], index=run_ids
        ).reindex(ids).to_numpy()

        if not np.array_equal(y_r, y):
            raise ValueError(f"Results not coherent, repetition {r}")

        P[:, r] = pd.Series(
            run["y_prob_oof"], index=run_ids
        ).reindex(ids).to_numpy()

    if not np.isin(y, [0, 1]).all() or np.unique(y).size != 2:
        raise ValueError("Binary results expected 0/1 in both classes")
    if not np.isfinite(P).all() or ((P < 0) | (P > 1)).any():
        raise ValueError("Wrong probabilities")

    return y, P


def summarize_repeated_cv(
    runs, threshold=0.30, n_boot=1000, seed=42
):
    y, P = align_oof_runs(runs)
    rng = np.random.default_rng(seed)

    # One metric row for each CV repetition
    per_run = np.vstack([
        metric_values(y, P[:, r], threshold)
        for r in range(P.shape[1])
    ])

    mean = per_run.mean(axis=0)
    std = (
        per_run.std(axis=0, ddof=1)
        if P.shape[1] > 1
        else np.full(len(METRIC_NAMES), np.nan)
    )

    # Bootstrap: subject resampling, keeping joined predictions in all repetitions
    boot_means = []

    with tqdm(total=n_boot,initial=len(boot_means),desc="Bootstrap",unit="sample",) as pbar:
        while len(boot_means) < n_boot:
            idx = rng.integers(0, len(y), len(y))
            y_b = y[idx]

            # AUC e AUPRC require both classes
            if np.unique(y_b).size < 2:
                continue

            P_b = P[idx, :]
            scores_b = np.vstack([
                metric_values(y_b, P_b[:, r], threshold)
                for r in range(P_b.shape[1])
            ])
            boot_means.append(scores_b.mean(axis=0))
            pbar.update(1)
        
    ci_low, ci_high = np.percentile(
        np.asarray(boot_means), [2.5, 97.5], axis=0
    )

    summary = pd.DataFrame({
        "mean": mean,
        "std_between_runs": std,
        "ci95_low": ci_low,
        "ci95_high": ci_high,
    }, index=METRIC_NAMES)

    per_run = pd.DataFrame(
        per_run,
        columns=METRIC_NAMES,
        index=pd.Index(range(len(runs)), name="iteration"),
    )

    return summary, per_run

In [ ]:
from joblib import Parallel, delayed, parallel_config

def summarize_method(m, results, n_iter, threshold, n_boot, seed):
    meth = "+".join(m)
    runs = [results[meth][i] for i in range(n_iter)]

    summary, per_run = summarize_repeated_cv(
        runs,
        threshold=threshold,
        n_boot=n_boot,
        seed=seed,
    )
    return meth, summary, per_run


with parallel_config(backend="loky", inner_max_num_threads=1):
    outputs = Parallel(n_jobs=-1)(
        delayed(summarize_method)(
            m=m,
            results=results,
            n_iter=args.iter,
            threshold=0.30,
            n_boot=1000,
            seed=42,
        )
        for m in args.methods
    )

summaries = {
    meth: summary
    for meth, summary, per_run in outputs
}

scores_by_run = {
    meth: per_run
    for meth, summary, per_run in outputs
}

Bootstrap: 100%|██████████| 1000/1000 [00:51<00:00, 19.29sample/s]


## Save metric as dataframe

In [5]:
from sklearn.metrics import confusion_matrix

rows = []

for m in args.methods:
    meth = "+".join(m)
    summary = summaries[meth]

    row = {"model": meth}

    for metric in METRIC_NAMES:
        s = summary.loc[metric]
        row[metric] = (
            f"{s['mean']:.3f}±{s['std_between_runs']:.3f} "
            f"[{s['ci95_low']:.3f}-{s['ci95_high']:.3f}]"
        )

    # Una confusion matrix per iterazione, poi media elemento per elemento
    runs = [results[meth][i] for i in range(args.iter)]
    y, P = align_oof_runs(runs)

    matrices = []
    for i in range(args.iter):
        predicted = (P[:, i] >= 0.30).astype(int)

        # .T produce il formato richiesto: [[TP FP] [FN TN]]
        cm = confusion_matrix(
            y, predicted, labels=[0, 1]
        )
        matrices.append(cm)

    mean_cm = np.mean(matrices, axis=0)

    row["Confusion matrix"] = (
        f"[[{mean_cm[0, 0]:.1f} {mean_cm[0, 1]:.1f}] "
        f"[{mean_cm[1, 0]:.1f} {mean_cm[1, 1]:.1f}]]"
    )

    rows.append(row)

formatted_results = pd.DataFrame(rows)
formatted_results.set_index("model", inplace=True)
formatted_results.to_csv(f"{args.results_dir}/result_table_{args.dataset}_ITER{args.iter}.csv")
formatted_results

,AUC,AUPRC,F1,PPV,Sensitivity,Specificity,NPV,MCC,Accuracy,Brier,Confusion matrix
model,,,,,,,,,,,
STATIC,0.602±0.018 [0.549-0.651],0.091±0.011 [0.067-0.134],0.130±0.014 [0.094-0.163],0.079±0.012 [0.055-0.104],0.408±0.133 [0.331-0.487],0.675±0.122 [0.654-0.692],0.945±0.006 [0.928-0.961],0.045±0.027 [0.001-0.084],0.658±0.106 [0.637-0.676],0.114±0.021 [0.105-0.123],[[538.3 259.7] [31.4 21.6]]
STATIC+BINARY,0.730±0.032 [0.674-0.785],0.238±0.044 [0.163-0.331],0.275±0.058 [0.189-0.351],0.278±0.067 [0.189-0.368],0.275±0.059 [0.193-0.365],0.952±0.009 [0.942-0.960],0.952±0.004 [0.937-0.965],0.228±0.063 [0.143-0.308],0.910±0.010 [0.894-0.923],0.059±0.003 [0.047-0.070],[[759.4 38.6] [38.4 14.6]]
STATIC+FREQUENCY,0.712±0.029 [0.653-0.767],0.219±0.047 [0.151-0.312],0.250±0.060 [0.168-0.327],0.279±0.079 [0.184-0.375],0.230±0.054 [0.155-0.312],0.959±0.012 [0.950-0.967],0.949±0.004 [0.934-0.963],0.207±0.066 [0.125-0.288],0.913±0.012 [0.898-0.927],0.059±0.003 [0.047-0.071],[[765.0 33.0] [40.8 12.2]]
STATIC+EVENT-CNT,0.711±0.026 [0.652-0.771],0.217±0.039 [0.156-0.310],0.278±0.058 [0.199-0.348],0.252±0.063 [0.173-0.330],0.319±0.078 [0.230-0.404],0.935±0.018 [0.923-0.945],0.954±0.005 [0.939-0.967],0.227±0.064 [0.149-0.303],0.896±0.017 [0.881-0.912],0.063±0.004 [0.053-0.075],[[746.0 52.0] [36.1 16.9]]
STATIC+TF-IDF,0.663±0.038 [0.613-0.713],0.161±0.043 [0.116-0.231],0.185±0.056 [0.122-0.245],0.196±0.071 [0.125-0.270],0.179±0.056 [0.120-0.248],0.949±0.013 [0.942-0.957],0.946±0.003 [0.930-0.961],0.134±0.062 [0.073-0.198],0.901±0.012 [0.887-0.915],0.063±0.003 [0.051-0.075],[[757.6 40.4] [43.5 9.5]]
STATIC+BOW,0.629±0.049 [0.589-0.669],0.133±0.040 [0.104-0.183],0.111±0.070 [0.073-0.146],0.194±0.098 [0.121-0.275],0.083±0.061 [0.055-0.112],0.978±0.009 [0.974-0.982],0.941±0.003 [0.925-0.957],0.088±0.067 [0.046-0.131],0.922±0.006 [0.907-0.938],0.060±0.002 [0.047-0.073],[[780.5 17.5] [48.6 4.4]]
STATIC+BiPadLSTM,0.631±0.032 [0.580-0.682],0.149±0.048 [0.108-0.213],0.203±0.061 [0.139-0.265],0.200±0.049 [0.130-0.269],0.211±0.079 [0.146-0.282],0.945±0.013 [0.938-0.953],0.948±0.005 [0.933-0.962],0.151±0.061 [0.087-0.217],0.899±0.010 [0.885-0.913],0.066±0.005 [0.053-0.079],[[754.1 43.9] [41.8 11.2]]
STATIC+Med2Vec,0.577±0.047 [0.538-0.617],0.123±0.028 [0.089-0.179],0.130±0.050 [0.071-0.192],0.253±0.075 [0.138-0.365],0.089±0.038 [0.048-0.136],0.983±0.005 [0.979-0.986],0.942±0.002 [0.926-0.958],0.117±0.052 [0.050-0.185],0.927±0.005 [0.911-0.942],0.062±0.005 [0.048-0.077],[[784.1 13.9] [48.3 4.7]]
STATIC+Dipole,0.588±0.032 [0.553-0.623],0.124±0.035 [0.090-0.176],0.197±0.070 [0.132-0.255],0.272±0.088 [0.179-0.364],0.157±0.061 [0.106-0.213],0.973±0.007 [0.968-0.978],0.946±0.004 [0.931-0.960],0.167±0.070 [0.101-0.232],0.922±0.006 [0.908-0.936],0.058±0.002 [0.044-0.072],[[776.4 21.6] [44.7 8.3]]


## To latex with split in N

In [6]:
args.dataset = "MIMIC"
import pandas as pd
formatted_results = pd.read_csv(f"{args.results_dir}/result_table_{args.dataset}_ITER{args.iter}.csv", index_col=0)
formatted_results

,AUC,AUPRC,F1,PPV,Sensitivity,Specificity,NPV,MCC,Accuracy,Brier,Confusion matrix
model,,,,,,,,,,,
STATIC,0.602±0.018 [0.549-0.651],0.091±0.011 [0.067-0.134],0.130±0.014 [0.094-0.163],0.079±0.012 [0.055-0.104],0.408±0.133 [0.331-0.487],0.675±0.122 [0.654-0.692],0.945±0.006 [0.928-0.961],0.045±0.027 [0.001-0.084],0.658±0.106 [0.637-0.676],0.114±0.021 [0.105-0.123],[[538.3 259.7] [31.4 21.6]]
STATIC+BINARY,0.730±0.032 [0.674-0.785],0.238±0.044 [0.163-0.331],0.275±0.058 [0.189-0.351],0.278±0.067 [0.189-0.368],0.275±0.059 [0.193-0.365],0.952±0.009 [0.942-0.960],0.952±0.004 [0.937-0.965],0.228±0.063 [0.143-0.308],0.910±0.010 [0.894-0.923],0.059±0.003 [0.047-0.070],[[759.4 38.6] [38.4 14.6]]
STATIC+FREQUENCY,0.712±0.029 [0.653-0.767],0.219±0.047 [0.151-0.312],0.250±0.060 [0.168-0.327],0.279±0.079 [0.184-0.375],0.230±0.054 [0.155-0.312],0.959±0.012 [0.950-0.967],0.949±0.004 [0.934-0.963],0.207±0.066 [0.125-0.288],0.913±0.012 [0.898-0.927],0.059±0.003 [0.047-0.071],[[765.0 33.0] [40.8 12.2]]
STATIC+EVENT-CNT,0.711±0.026 [0.652-0.771],0.217±0.039 [0.156-0.310],0.278±0.058 [0.199-0.348],0.252±0.063 [0.173-0.330],0.319±0.078 [0.230-0.404],0.935±0.018 [0.923-0.945],0.954±0.005 [0.939-0.967],0.227±0.064 [0.149-0.303],0.896±0.017 [0.881-0.912],0.063±0.004 [0.053-0.075],[[746.0 52.0] [36.1 16.9]]
STATIC+TF-IDF,0.663±0.038 [0.613-0.713],0.161±0.043 [0.116-0.231],0.185±0.056 [0.122-0.245],0.196±0.071 [0.125-0.270],0.179±0.056 [0.120-0.248],0.949±0.013 [0.942-0.957],0.946±0.003 [0.930-0.961],0.134±0.062 [0.073-0.198],0.901±0.012 [0.887-0.915],0.063±0.003 [0.051-0.075],[[757.6 40.4] [43.5 9.5]]
STATIC+BOW,0.629±0.049 [0.589-0.669],0.133±0.040 [0.104-0.183],0.111±0.070 [0.073-0.146],0.194±0.098 [0.121-0.275],0.083±0.061 [0.055-0.112],0.978±0.009 [0.974-0.982],0.941±0.003 [0.925-0.957],0.088±0.067 [0.046-0.131],0.922±0.006 [0.907-0.938],0.060±0.002 [0.047-0.073],[[780.5 17.5] [48.6 4.4]]
STATIC+BiPadLSTM,0.631±0.032 [0.580-0.682],0.149±0.048 [0.108-0.213],0.203±0.061 [0.139-0.265],0.200±0.049 [0.130-0.269],0.211±0.079 [0.146-0.282],0.945±0.013 [0.938-0.953],0.948±0.005 [0.933-0.962],0.151±0.061 [0.087-0.217],0.899±0.010 [0.885-0.913],0.066±0.005 [0.053-0.079],[[754.1 43.9] [41.8 11.2]]
STATIC+Med2Vec,0.577±0.047 [0.538-0.617],0.123±0.028 [0.089-0.179],0.130±0.050 [0.071-0.192],0.253±0.075 [0.138-0.365],0.089±0.038 [0.048-0.136],0.983±0.005 [0.979-0.986],0.942±0.002 [0.926-0.958],0.117±0.052 [0.050-0.185],0.927±0.005 [0.911-0.942],0.062±0.005 [0.048-0.077],[[784.1 13.9] [48.3 4.7]]
STATIC+Dipole,0.588±0.032 [0.553-0.623],0.124±0.035 [0.090-0.176],0.197±0.070 [0.132-0.255],0.272±0.088 [0.179-0.364],0.157±0.061 [0.106-0.213],0.973±0.007 [0.968-0.978],0.946±0.004 [0.931-0.960],0.167±0.070 [0.101-0.232],0.922±0.006 [0.908-0.936],0.058±0.002 [0.044-0.072],[[776.4 21.6] [44.7 8.3]]


In [7]:
import pandas as pd
from scripts.utils import highlight_best_values
nsplit = 4

# Divide le colonne in tre gruppi equilibrati
n = len(formatted_results.columns)
base, resto = divmod(n, nsplit)
dimensioni = [base + (i < resto) for i in range(nsplit)]

blocchi = []
inizio = 0
for dimensione in dimensioni:
    blocchi.append(formatted_results.iloc[:, inizio:inizio + dimensione].copy())
    inizio += dimensione

# Usa il primo blocco per definire le colonne della tabella finale
larghezza = max(dimensioni)
colonne_finali = (
    ["Model"]
    + blocchi[0].columns.tolist()
    + [""] * (larghezza - blocchi[0].shape[1])
)

pezzi = []
for i, blocco in enumerate(blocchi):
    if i > 0:
        intestazione = (
            ["Model"]
            + blocco.columns.tolist()
            + [""] * (larghezza - blocco.shape[1])
        )
        pezzi.append(pd.DataFrame([intestazione], columns=colonne_finali))

    righe = blocco.rename_axis("Model").reset_index()
    righe.columns = colonne_finali[:righe.shape[1]]
    righe = righe.reindex(columns=colonne_finali, fill_value="")
    pezzi.append(righe)

tabella = pd.concat(pezzi, ignore_index=True)
print(highlight_best_values(tabella.to_latex(index=False, escape=True)))

\begin{tabular}{llll}
\toprule
Model & AUC & AUPRC & F1 \\
\midrule
STATIC & 0.602±0.018 [0.549-0.651] & 0.091±0.011 [0.067-0.134] & 0.130±0.014 [0.094-0.163] \\
STATIC+BINARY & \textbf{0.730}±0.032 [0.674-0.785] & \textbf{0.238}±0.044 [0.163-0.331] & 0.275±0.058 [0.189-0.351] \\
STATIC+FREQUENCY & 0.712±0.029 [0.653-0.767] & 0.219±0.047 [0.151-0.312] & 0.250±0.060 [0.168-0.327] \\
STATIC+EVENT-CNT & 0.711±0.026 [0.652-0.771] & 0.217±0.039 [0.156-0.310] & \textbf{0.278}±0.058 [0.199-0.348] \\
STATIC+TF-IDF & 0.663±0.038 [0.613-0.713] & 0.161±0.043 [0.116-0.231] & 0.185±0.056 [0.122-0.245] \\
STATIC+BOW & 0.629±0.049 [0.589-0.669] & 0.133±0.040 [0.104-0.183] & 0.111±0.070 [0.073-0.146] \\
STATIC+BiPadLSTM & 0.631±0.032 [0.580-0.682] & 0.149±0.048 [0.108-0.213] & 0.203±0.061 [0.139-0.265] \\
STATIC+Med2Vec & 0.577±0.047 [0.538-0.617] & 0.123±0.028 [0.089-0.179] & 0.130±0.050 [0.071-0.192] \\
STATIC+Dipole & 0.588±0.032 [0.553-0.623] & 0.124±0.035 [0.090-0.176] & 0.197±0.070 [0.132-0.255]